### Phrase-level frequency count of SAR words in PR title, body, and comments

In [ ]:
import pandas as pd
import re
from collections import defaultdict, Counter

from google.colab import drive
drive.mount('/content/drive')

import sys
directory = '/content/drive/Shared drives/ICS 691G Research Project/Research Project/Data Analysis'
sys.path.append(directory)

import bug_fix_list
import internal_list
import external_list
import functional_list
import code_smell_list

bug_words = bug_fix_list.bug_words
internal_words = internal_list.internal_words
external_words = external_list.external_words
functional_words = functional_list.functional_words
smell_words = code_smell_list.smell_words

Mounted at /content/drive


In [ ]:
print('Starting script.')
pull_request_df = pd.read_parquet("hf://datasets/hao-li/AIDev/pull_request.parquet")
comments_df = pd.read_parquet("hf://datasets/hao-li/AIDev/pr_comments.parquet")
pr_task_type_df = pd.read_parquet("hf://datasets/hao-li/AIDev/pr_task_type.parquet")
print('Finished querying parquets.')
pull_request_df.rename(columns={'id': 'pr_id'}, inplace=True)

Starting script.


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Finished querying parquets.


In [ ]:
# need body from pr_comments
combined_df = pull_request_df.merge(
    comments_df[['id', 'body']].add_prefix('comment_'),
    how='left',
    left_on='pr_id',
    right_on='comment_id',
    suffixes=('', '_task')
)
combined_df.drop(columns=['comment_id'])

# only look at refactor
combined_df = combined_df.merge(
    pr_task_type_df[['id', 'type']],
    how='left',
    left_on='pr_id',
    right_on='id'
)
combined_df.rename(columns={'type': 'pr_type'}, inplace=True)
combined_df = combined_df.loc[combined_df['pr_type'].str.contains('refactor', na=False)].copy()


print(f"Length of combined_df: {len(combined_df)}")
# print(combined_df)

Length of combined_df: 2288


In [ ]:
print("Counting SAR phrase occurrences...")

def to_regex_pattern(word):
    return re.escape(word).replace(r'\*', '.*')

for phrase in bug_words:
    pattern = re.compile(to_regex_pattern(phrase), re.IGNORECASE)
    combined_df[phrase] = (
        combined_df['body'].str.contains(pattern, na=False) |
        combined_df['comment_body'].str.contains(pattern, na=False) |
        combined_df['title'].str.contains(pattern, na=False)
    )

for phrase in internal_words:
    pattern = re.compile(to_regex_pattern(phrase), re.IGNORECASE)
    combined_df[phrase] = (
        combined_df['body'].str.contains(pattern, na=False) |
        combined_df['comment_body'].str.contains(pattern, na=False) |
        combined_df['title'].str.contains(pattern, na=False)
    )

for phrase in external_words:
    pattern = re.compile(to_regex_pattern(phrase), re.IGNORECASE)
    combined_df[phrase] = (
        combined_df['body'].str.contains(pattern, na=False) |
        combined_df['comment_body'].str.contains(pattern, na=False) |
        combined_df['title'].str.contains(pattern, na=False)
    )

for phrase in functional_words:
    pattern = re.compile(to_regex_pattern(phrase), re.IGNORECASE)
    combined_df[phrase] = (
        combined_df['body'].str.contains(pattern, na=False) |
        combined_df['comment_body'].str.contains(pattern, na=False) |
        combined_df['title'].str.contains(pattern, na=False)
    )

for phrase in smell_words:
    pattern = re.compile(to_regex_pattern(phrase), re.IGNORECASE)
    combined_df[phrase] = (
        combined_df['body'].str.contains(pattern, na=False) |
        combined_df['comment_body'].str.contains(pattern, na=False) |
        combined_df['title'].str.contains(pattern, na=False)
    )



unique_prs = combined_df.drop_duplicates(subset=['pr_id'])

total_requests = (
    unique_prs
    .groupby(['agent'])
    .size()
    .reset_index(name='total_requests')
)

sar_bug_words = (
    unique_prs
    .groupby(['agent'])
    [bug_words]
    .sum()
    .reset_index()
)

sar_internal_words = (
    unique_prs
    .groupby(['agent'])
    [internal_words]
    .sum()
    .reset_index()
)

sar_external_words = (
    unique_prs
    .groupby(['agent'])
    [external_words]
    .sum()
    .reset_index()
)

sar_functional_words = (
    unique_prs
    .groupby(['agent'])
    [functional_words]
    .sum()
    .reset_index()
)

sar_smell_words = (
    unique_prs
    .groupby(['agent'])
    [smell_words]
    .sum()
    .reset_index()
)

summary = (
    total_requests
    .merge(sar_bug_words, on=['agent'], how='left')
    .merge(sar_internal_words, on=['agent'], how='left')
    .merge(sar_external_words, on=['agent'], how='left')
    .merge(sar_functional_words, on=['agent'], how='left')
    .merge(sar_smell_words, on=['agent'], how='left')
)

print(summary)

# Save output
output_path = "sar_rq1_phrase_frequency_table.csv"
summary.to_csv(output_path)
print(f"Saved table: {output_path}")

Counting SAR phrase occurrences...
['Minor fixes', 'Get rid of', 'Additional fixes', 'Handle']
['Decoupling', 'Reduce complexity', 'Chang* inheritance', 'Better encapsulation', 'Remov* dependency']
['More flexibility', 'Better readability', 'Improve readability', 'Increase readability', 'Readability improvements', 'Improv* performance', 'More manageable', 'More efficient*', 'Performance enhancement', 'More compatible', 'Improve usability', 'To be more robust', 'Easier to understand']
['New module', 'Fix some GUI', 'Move functionality', 'UI changes']
['Avoid code duplication', 'Avoid duplicate code', 'Code duplication removed', 'Delet* duplicate code', 'Eliminate duplicate code', 'Reduce code duplication', 'Refactored duplicate code', 'Remov* code duplication', 'Remov* duplicate code']
          agent  total_requests  Minor fixes  Get rid of  Additional fixes  \
0   Claude_Code              26            0           0                 0   
1       Copilot             301            0    

### Disregard below

In [ ]:
print("Counting SAR phrase occurrences...")

# Assemble regex patterns
all_words = bug_words + internal_words + external_words + functional_words + smell_words

def to_regex_pattern(word):
    # Escape everything except the wildcard symbol *
    return re.escape(word).replace(r'\*', '.*')

compiled_patterns = {
    phrase: re.compile(to_regex_pattern(phrase), re.IGNORECASE)
    for phrase in all_words
}

# Initialize nested counter: {agent: {phrase: count}}
agent_phrase_counts = defaultdict(Counter)
counter = 0
# Loop through each PR
for idx, row in combined_df.iterrows():
    agent = row["agent"]
    if pd.isna(agent):
        continue

    # Combine text fields (PR title, body, comment body)
    text_parts = [row.get("title", ""), row.get("body", ""), row.get("comment_body", "")]
    text = " ".join(str(t) for t in text_parts if isinstance(t, str)).lower()

    if not text.strip():
        continue

    # For each SAR phrase, count occurrences in this PR text
    match_found = False

    for phrase, pattern in compiled_patterns.items():
        counter += 1;
        print(pattern.findall(text))
        count = len(pattern.findall(text)) # the amount of times a SAR phrase appears in the title, body, or comment
        if count > 0:
            agent_phrase_counts[agent][phrase] += 1
            # print(agent, " ", phrase, agent_phrase_counts[agent][phrase], "")
            match_found = True

    # Track whether this PR had *any* SAR phrase
    if match_found:
        agent_phrase_counts[agent]["Messages with SAR"] += 1

print("Finished counting SAR phrase occurrences.")
print("counter:", counter)
# Convert results to DataFrame
frequency_df = pd.DataFrame(agent_phrase_counts).T.fillna(0).astype(int)
frequency_df.index.name = "Agent"

# Add total messages per agent
total_messages = combined_df.groupby("agent")["pr_id"].nunique()
frequency_df["Total Messages"] = frequency_df.index.map(total_messages).fillna(0).astype(int)

# Compute percentage of messages that contained any SAR phrase
frequency_df["% with SAR"] = (
    frequency_df["Messages with SAR"] / frequency_df["Total Messages"] * 100
).round(2)

# Sort columns for readability
cols = (
    ["Messages with SAR", "Total Messages", "% with SAR"]
    + [c for c in frequency_df.columns if c not in ["Messages with SAR", "Total Messages", "% with SAR"]]
)
frequency_df = frequency_df[cols]

# Save output
output_path = "sar_rq1_phrase_frequency_table.csv"
frequency_df.to_csv(output_path)
print(f"Saved table: {output_path}")

# Display summary
print(frequency_df.head())

Streaming output truncated to the last 5000 lines.
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
['handle']
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
['reduce code duplication']
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
['handle']
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
['handle', 'handle']
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]

In [ ]:
print("Counting SAR phrase occurrences...")

# Initialize counter structure
agent_phrase_counts = defaultdict(Counter)

# Process rows in chunks to save RAM
chunk_size = 50000  # tune for your RAM; 50k is safe for 16GB machines
for start in range(0, len(combined_df), chunk_size):
    end = start + chunk_size
    chunk = combined_df.iloc[start:end]

    for _, row in chunk.iterrows():
        agent = row["agent"]
        if pd.isna(agent):
            continue
        text_parts = [row.get("body", "")]
        text = " ".join(str(t) for t in text_parts if isinstance(t, str))
        if not text.strip():
            continue

        text_lower = text.lower()
        for phrase, pattern in compiled_patterns.items():
            # count occurrences of phrase in this text
            count = len(pattern.findall(text_lower))
            if count > 0:
                agent_phrase_counts[agent][phrase] += count

    print(f"  Processed rows {start:,}-{end:,}")

print("Counting complete. Converting to DataFrame...")

# Convert to table
frequency_df = pd.DataFrame(agent_phrase_counts).T.fillna(0).astype(int)
frequency_df.index.name = "Agent"

# Sum of all phrase counts per agent
frequency_df["Messages with SAR"] = frequency_df.sum(axis=1)

# Compute total number of commit messages per agent, independent of matches
print("Computing overall statistics...")
agent_total_messages = combined_df.groupby("agent")["message"].count().astype(int)
frequency_df["Total Messages"] = frequency_df.index.map(agent_total_messages)

# Calculate the percentage
frequency_df["% with SAR"] = (
    (frequency_df["Messages with SAR"] / frequency_df["Total Messages"]) * 100
).round(2)

# Save output
frequency_df.to_csv("phrase_frequency_.csv")
print("Saved table: phrase_frequency_.csv")

# Show preview
print(frequency_df.head())

Counting SAR phrase occurrences...


NameError: name 'defaultdict' is not defined

In [ ]:
import pandas as pd
import re
from collections import defaultdict, Counter
from bug_fix_list import bug_words
from internal_list import internal_words
from external_list import external_words
from functional_list import functional_words
from code_smell_list import smell_words

print("Starting memory-efficient SAR keyword frequency script...")

# Load dataset files from Hugging Face
print("Loading parquet files...")
all_pr_df = pd.read_parquet("hf://datasets/hao-li/AIDev/all_pull_request.parquet")
pr_commit_details_df = pd.read_parquet("hf://datasets/hao-li/AIDev/pr_commit_details.parquet")
print("Parquet files loaded.")

# Merge metadata into one frame
print("Merging datasets...")
combined_df = pd.merge(
    all_pr_df[['id', 'agent']],
    pr_commit_details_df[['pr_id', 'message']],
    left_on='id',
    right_on='pr_id',
    how='left'
)

# Drop the redundant key column from pr_commit_details
combined_df = combined_df.drop(columns=['pr_id'])

# Remove rows where 'message' is NaN
combined_df = combined_df.dropna(subset=['message'])

print(combined_df)

print("Datasets merged.")

Starting memory-efficient SAR keyword frequency script...
Loading parquet files...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Parquet files loaded.
Merging datasets...
                 id        agent  \
26       3264933329  Claude_Code   
27       3264933329  Claude_Code   
28       3264933329  Claude_Code   
38       3265118634  Claude_Code   
39       3265118634  Claude_Code   
...             ...          ...   
1611128  2858527610        Devin   
1611129  2858527610        Devin   
1611130  2858527610        Devin   
1611131  2858527610        Devin   
1611132  2858527610        Devin   

                                                   message  
26       fix: Wait for all partitions in load_collectio...  
27       fix: Wait for all partitions in load_collectio...  
28       fix: Wait for all partitions in load_collectio...  
38       ファイルパス参照を相対パスに統一し、doc/からdocs/に統一\n\n- commands...  
39       ファイルパス参照を相対パスに統一し、doc/からdocs/に統一\n\n- commands...  
...                                                    ...  
1611128  Add doxygen documentation for public APIs\n\nC...  
1611129  Add newlines between doxygen

In [ ]:
# Combine all patterns
all_patterns = list(set(
    bug_words +
    internal_words +
    external_words +
    functional_words +
    smell_words
))

# Precompile regex patterns for each phrase (faster lookups)
compiled_patterns = {phrase: re.compile(re.escape(phrase), re.IGNORECASE) for phrase in all_patterns}

# Initialize counter structure
agent_phrase_counts = defaultdict(Counter)

print("Counting SAR phrase occurrences...")

# Process rows in chunks to save RAM
chunk_size = 50000  # tune for your RAM; 50k is safe for 16GB machines
for start in range(0, len(combined_df), chunk_size):
    end = start + chunk_size
    chunk = combined_df.iloc[start:end]

    for _, row in chunk.iterrows():
        agent = row["agent"]
        if pd.isna(agent):
            continue
        text_parts = [row.get("message", "")]
        text = " ".join(str(t) for t in text_parts if isinstance(t, str))
        if not text.strip():
            continue

        text_lower = text.lower()
        for phrase, pattern in compiled_patterns.items():
            # count occurrences of phrase in this text
            count = len(pattern.findall(text_lower))
            if count > 0:
                agent_phrase_counts[agent][phrase] += count

    print(f"  Processed rows {start:,}-{end:,}")

print("Counting complete. Converting to DataFrame...")

# Convert to table
frequency_df = pd.DataFrame(agent_phrase_counts).T.fillna(0).astype(int)
frequency_df.index.name = "Agent"

# Sum of all phrase counts per agent
frequency_df["Messages with SAR"] = frequency_df.sum(axis=1)

# Compute total number of commit messages per agent, independent of matches
print("Computing overall statistics...")
agent_total_messages = combined_df.groupby("agent")["message"].count().astype(int)
frequency_df["Total Messages"] = frequency_df.index.map(agent_total_messages)

# Calculate the percentage
frequency_df["% with SAR"] = (
    (frequency_df["Messages with SAR"] / frequency_df["Total Messages"]) * 100
).round(2)

# Save output
frequency_df.to_csv("phrase_frequency_.csv")
print("Saved table: phrase_frequency_.csv")

# Show preview
print(frequency_df.head())


Counting SAR phrase occurrences...
  Processed rows 0-50,000
  Processed rows 50,000-100,000
  Processed rows 100,000-150,000
  Processed rows 150,000-200,000
  Processed rows 200,000-250,000
  Processed rows 250,000-300,000
  Processed rows 300,000-350,000
  Processed rows 350,000-400,000
  Processed rows 400,000-450,000
  Processed rows 450,000-500,000
  Processed rows 500,000-550,000
  Processed rows 550,000-600,000
  Processed rows 600,000-650,000
  Processed rows 650,000-700,000
  Processed rows 700,000-750,000
Counting complete. Converting to DataFrame...
Computing overall statistics...
Saved table: phrase_frequency_.csv
              Handle  Improve readability  Improved performance  \
Agent                                                             
Claude_Code     1428                   17                   127   
Copilot         1617                    4                     0   
OpenAI_Codex    6261                   13                     9   
Cursor           376          